In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [ ]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("threat_analysis")
    .getOrCreate()
    )

In [ ]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

## Validação da Ameaça

In [ ]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
df_threat = spark.read.parquet(threat_dataset_path)

In [ ]:
def build_aggregated_df(
    df,
    group_cols,
    agg_cols,
    agg_func,
    agg_prefix
):
    """
    Agrega um DataFrame utilizando a função de agregação desejada.
    """

    # expressões de agregação com o prefixo conforme a agregação realizada (ex: média -> avg, máximo -> max)
    agg_exprs = [
        F.round(agg_func(F.col(c)), 3).alias(f"{agg_prefix}_{c}")
        for c in agg_cols
    ]

    # DataFrame agregado
    df_agg = (
        df
        .groupBy(*group_cols)
        .agg(*agg_exprs)
    )

    # nomes das colunas agregadas
    agg_cols_result = [
        f"{agg_prefix}_{c}"
        for c in agg_cols
    ]

    return agg_cols_result, df_agg


def join_match_stats(df, df_match_stats):

    # adiciona estatísticas da partida
    return (
        df.join(
            df_match_stats,
            on=[
                "date",
                "homeTeamName",
                "opponentTeamName"
            ],
            how="left"
        )
    )


def build_side_df(df, is_home, avg_cols):

    # filtra mandante ou visitante
    side_filter = F.col('homeTeam') if is_home else ~F.col('homeTeam')

    # mapeamento das colunas conforme o lado
    cols = {
        "teamName": "homeTeamName" if is_home else "opponentTeamName",
        "win": (F.col("FTR") == ('H' if is_home else 'A')),
        "goals": "FTHG" if is_home else "FTAG",
        "shots": "HS" if is_home else "AS",
        "shots_target": "HST" if is_home else "AST",
        "avg_win_odds": "AvgH" if is_home else "AvgA",
    }

    # colunas fixas
    fixed_select = [
        'competitionName',
        'season',
        'gameId',
        "date",
        F.col(cols["teamName"]).alias("teamName"),
    ]

    # métricas agregadas
    avg_select = [F.col(c) for c in avg_cols]

    # estatísticas da partida
    result_select = [
        cols["win"].alias("win"),
        F.col(cols["goals"]).alias("goals"),
        F.col(cols["shots"]).alias("shots"),
        F.col(cols["shots_target"]).alias("shots_target"),
        F.col(cols["avg_win_odds"]).alias("avg_win_odds"),
    ]

    return (
        df
        .filter(side_filter)
        .select(*fixed_select, *avg_select, *result_select)
    )

In [ ]:
def plot_correlation_heatmap(df_agg, corr_cols):
    """
    Constrói o df agregado (via build_aggregated_df), converte para pandas,
    calcula a matriz de correlação das colunas em `corr_cols` e plota um
    heatmap com Plotly.
    """

    df_agg_pd = df_agg.toPandas()

    corr = df_agg_pd[corr_cols].corr()

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale='RdBu', 
            zmin=-1, 
            zmax=1,
            text=corr.round(2).values,
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title="Matriz de Correlação",
        template="simple_white",
        width=1500,
        height=900,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

#### 1.2. Preparação da base de odds

In [ ]:
# base de odds da Premier League 2022-2023
match_stats_22_23_path = str(data_folder_path / "match_stats" / "PL_22_23.csv")

df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(match_stats_22_23_path , sep=',')

In [ ]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23
    .select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
    )
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
    .sort('date')
)

df_pl_match_stats_22_23_filtrado_mapped.show()

### 2. Criação das bases agregadas e teste de correlação

### 2.1. Por Time-Partida

In [ ]:
group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam"
]

# zonas do campo consideradas nas colunas geradas no target_engineering
zone_suffixes = ['', '_half', '_third_2', '_third_3']

agg_cols = []
for suffix in zone_suffixes:
    agg_cols += [
        f"threat_score{suffix}",
        f"progression_dist{suffix}_norm",
        f"total_players{suffix}_norm",
        f"atk_def_advantage{suffix}_norm"
    ]

avg_cols, df_agg = build_aggregated_df(
    df=df_threat,
    group_cols=group_cols,
    agg_cols=agg_cols,
    agg_func=F.mean,
    agg_prefix="avg"
)

df_agg = join_match_stats(
    df_agg,
    df_pl_match_stats_22_23_filtrado_mapped
)

df_team_match = (
    build_side_df(df_agg, True, avg_cols)
    .unionByName(
        build_side_df(df_agg, False, avg_cols)
    )
)

corr_cols = (
    df_team_match
    .drop(*(group_cols + ["teamName"]))
    .columns
)

df_team_match.show(5)

plot_correlation_heatmap(df_team_match, corr_cols)

### 2.2. Por Time-Partida-Ciclo de Posse

In [ ]:
possession_group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam",
    "possession_id"
]

max_cols, df_possession = build_aggregated_df(
    df=df_threat,
    group_cols=possession_group_cols,
    agg_cols=agg_cols,
    agg_func=F.max,
    agg_prefix="max"
)

team_group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam"
]

avg_cols, df_agg = build_aggregated_df(
    df=df_possession,
    group_cols=team_group_cols,
    agg_cols=max_cols,
    agg_func=F.mean,
    agg_prefix="avg"
)

df_agg = join_match_stats(
    df_agg,
    df_pl_match_stats_22_23_filtrado_mapped
)

df_team_match = (
    build_side_df(df_agg, True, avg_cols)
    .unionByName(
        build_side_df(df_agg, False, avg_cols)
    )
)

corr_cols = (
    df_team_match
    .drop(*(group_cols + ["teamName"]))
    .columns
)

df_team_match.show(5)

plot_correlation_heatmap(df_team_match, corr_cols)